# DNA-aware stochastic model (auto-run)

This notebook includes:
- DNA → HDR rate Hill model
- Skew-aware stochastic simulator
- Monte Carlo evaluation
- Synthetic dataset generator
- Surrogate regression (Ridge)
- Experimental parameter suggestion + CSV export

When you run the notebook top-to-bottom, it will **save plots + a CSV** into the same folder as the notebook (or your working directory).


In [1]:
import os
# If you run this in the same folder as the module, this import should work.
# Otherwise, update the path below.
import sys
sys.path.append('.')
sys.path.append('/mnt/data')  # helpful on the cluster / sandbox

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sge_model_skew_dna_reexport as mod

print('Imported:', mod.__file__)


Imported: /Users/ds39/PycharmProjects/MAVE_Project/Simulation_Prediction_modelling/sge_model_skew_dna_reexport.py


In [2]:
# -----------------------------
# Settings (edit these)
# -----------------------------
OUTDIR = '.'  # change to e.g. './outputs'
os.makedirs(OUTDIR, exist_ok=True)

# Simulation settings
CELLS_TRANSFECTED = 200_000
LIBRARY_SIZE = 2000
READS = 1_000_000
SKEW_SIGMA = 0.5

# Sweep grid
HDR_VALUES = np.linspace(100, 2000, 8)
SGRNA_VALUES = np.linspace(50, 1500, 8)

# Monte Carlo reps per grid point (increase for smoother plots)
N_REPS_GRID = 30

# Suggestion search
TARGET_DROPOUT = 0.02
N_CANDIDATES = 2000

print('OUTDIR =', OUTDIR)


OUTDIR = .


In [3]:
# -----------------------------
# 1) Grid sweep (HDR x sgRNA)
# -----------------------------
dropout_grid = np.zeros((len(HDR_VALUES), len(SGRNA_VALUES)))
p10_grid = np.zeros_like(dropout_grid)

for i, h in enumerate(HDR_VALUES):
    for j, s in enumerate(SGRNA_VALUES):
        res = mod.run_monte_carlo(
            n_reps=N_REPS_GRID,
            hdr_ng=float(h),
            sgrna_ng=float(s),
            skew_sigma=SKEW_SIGMA,
            cells_transfected=CELLS_TRANSFECTED,
            library_size=LIBRARY_SIZE,
            reads=READS,
        )
        dropout_grid[i, j] = res['dropout_mean']
        p10_grid[i, j] = res['p10_reads_mean']

print('Sweep done. Dropout range:', dropout_grid.min(), 'to', dropout_grid.max())


Sweep done. Dropout range: 0.0 to 0.41246666666666665


In [4]:
# -----------------------------
# 2) Plots + save figures
# -----------------------------
dropout_vs_hdr = dropout_grid.mean(axis=1)
p10_vs_hdr = p10_grid.mean(axis=1)

# Plot A: dropout vs HDR (sgRNA averaged)
plt.figure()
plt.plot(HDR_VALUES, dropout_vs_hdr)
plt.xlabel('HDR (ng)')
plt.ylabel('Mean dropout')
plt.title('Mean dropout vs HDR (sgRNA averaged)')
plt.grid(True)
plt.tight_layout()
plot_dropout_vs_hdr = os.path.join(OUTDIR, 'plot_dropout_vs_hdr.png')
plt.savefig(plot_dropout_vs_hdr, dpi=200)
plt.close()

# Plot B: p10 reads vs HDR (sgRNA averaged)
plt.figure()
plt.plot(HDR_VALUES, p10_vs_hdr)
plt.xlabel('HDR (ng)')
plt.ylabel('Mean p10 reads')
plt.title('Mean p10 reads vs HDR (sgRNA averaged)')
plt.grid(True)
plt.tight_layout()
plot_p10_vs_hdr = os.path.join(OUTDIR, 'plot_p10_vs_hdr.png')
plt.savefig(plot_p10_vs_hdr, dpi=200)
plt.close()

# Plot C: heatmap dropout over HDR x sgRNA
plt.figure()
plt.imshow(dropout_grid, aspect='auto', origin='lower')
plt.xticks(ticks=np.arange(len(SGRNA_VALUES)), labels=[f'{v:.0f}' for v in SGRNA_VALUES], rotation=45)
plt.yticks(ticks=np.arange(len(HDR_VALUES)), labels=[f'{v:.0f}' for v in HDR_VALUES])
plt.xlabel('sgRNA (ng)')
plt.ylabel('HDR (ng)')
plt.title('Dropout heatmap (HDR x sgRNA)')
plt.colorbar(label='Mean dropout')
plt.tight_layout()
dropout_heatmap = os.path.join(OUTDIR, 'dropout_heatmap.png')
plt.savefig(dropout_heatmap, dpi=200)
plt.close()

print('Saved figures:')
print(' -', plot_dropout_vs_hdr)
print(' -', plot_p10_vs_hdr)
print(' -', dropout_heatmap)


Saved figures:
 - ./plot_dropout_vs_hdr.png
 - ./plot_p10_vs_hdr.png
 - ./dropout_heatmap.png


In [5]:
# -----------------------------
# 3) Synthetic dataset + surrogate regression
# -----------------------------
X, y = mod.build_synthetic_dataset(n_samples=300)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

surrogate = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
surrogate.fit(X, y)

print('Surrogate R^2 on training set:', surrogate.score(X, y))


Surrogate R^2 on training set: 0.3448844679235513


In [6]:
# -----------------------------
# 4) Suggest experiments + export CSV
# -----------------------------
suggestions = mod.suggest_experiments(target_dropout=TARGET_DROPOUT, n_candidates=N_CANDIDATES)
if len(suggestions) == 0:
    print('No suggestions at TARGET_DROPOUT=', TARGET_DROPOUT, '— relaxing to 0.05')
    suggestions = mod.suggest_experiments(target_dropout=0.05, n_candidates=N_CANDIDATES)

df_sugg = pd.DataFrame(suggestions)
csv_path = os.path.join(OUTDIR, 'suggested_experiments.csv')
df_sugg.to_csv(csv_path, index=False)

print('Saved CSV:', csv_path)
print('Number of suggested settings:', len(df_sugg))
df_sugg.head(10)


Saved CSV: ./suggested_experiments.csv
Number of suggested settings: 1635


,HDR_ng,sgRNA_ng,skew_sigma,dropout,p10_reads
0,977.034080,745.667193,1.156615,0.000300,281.840
1,347.092364,908.746158,0.483039,0.000000,1086.515
2,204.864800,1150.828801,0.559684,0.003150,816.540
3,1593.189067,628.059105,0.999974,0.000050,419.625
4,617.990498,378.110584,1.325191,0.004200,178.700
5,549.476622,175.692982,0.987811,0.008525,372.595
6,942.132973,1150.772822,0.596215,0.000000,948.545
7,1774.529339,800.578189,0.110248,0.000000,2096.635
8,1377.983089,637.545195,1.399646,0.001700,151.750
9,478.703825,775.162325,0.665690,0.000000,804.245


## Notes
- If you want faster runs, reduce `N_REPS_GRID` and `N_CANDIDATES`.
- If you want more stable results, increase `N_REPS_GRID` (and optionally the internal Monte Carlo reps in the module).
- To change the DNA→HDR mapping, edit `dna_to_hdr_rate()` in `sge_model_skew_dna_reexport.py`.
